# Cross-Validation in Machine Learning

Cross-validation is a resampling technique used to assess the performance of a machine learning model on unseen data. It helps in:
- Avoiding overfitting
- Selecting the best model
- Tuning hyperparameters

Every method below is executed on real data — including the ways each one fails.

## 1. What is Cross-Validation?

The goal is an honest estimate of how a model will behave on data it has never seen. The *holdout* method (Section 2) gives you one such estimate, and we will start by measuring how unreliable that single estimate can be — the result is the entire motivation for cross-validation.

## 2. Holdout Method

The simplest form of cross-validation, where the dataset is split into:
- **Training Set**: Used to train the model.
- **Test Set**: Used to evaluate model performance.

We use the real breast-cancer dataset (569 tumors, 30 features) and a logistic regression.

In [1]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

X, y = load_breast_cancer(return_X_y=True)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = LogisticRegression(max_iter=5000)
model.fit(X_train, y_train)
print("Test accuracy:", round(model.score(X_test, y_test), 3))

Test accuracy: 0.965


### How much can one split lie?

That score depends on *which 20% of the rows* landed in the test set. Let's repeat the exact same experiment with 30 different random splits and watch the score move.

In [2]:
import numpy as np

scores = []
for random_state in range(30):
    Xtr, Xte, ytr, yte = train_test_split(
        X, y, test_size=0.2, random_state=random_state, stratify=y)
    m = LogisticRegression(max_iter=5000).fit(Xtr, ytr)
    scores.append(m.score(Xte, yte))
scores = np.array(scores)

print(f"mean   : {scores.mean():.3f}")
print(f"std    : {scores.std():.3f}")
print(f"best   : {scores.max():.3f}")
print(f"worst  : {scores.min():.3f}")
print(f"spread : {scores.max() - scores.min():.3f} accuracy points across 30 splits")

mean   : 0.953
std    : 0.012
best   : 0.974
worst  : 0.930
spread : 0.044 accuracy points across 30 splits


**What just happened.** The same model on the same data scored anywhere from **0.930 to 0.974** — a **4.4-point spread** decided purely by which rows happened to fall in the test set. If you tuned a model and reported the lucky split's 0.974 while a colleague's identical model reported 0.930, you would both be telling the truth and both be wrong about the model. Cross-validation replaces *one coin flip* with *the average of k coin flips* — and reports the spread instead of hiding it.

## 3. K-Fold Cross-Validation

Divides the dataset into *k* equal folds. The model is trained on *k-1* folds and tested on the remaining fold. The process is repeated *k* times, and results are averaged.

$$
Error = \frac{1}{k} \sum_{i=1}^{k} Error_i
$$

In [3]:
from sklearn.model_selection import KFold, cross_val_score

kf = KFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(model, X, y, cv=kf, scoring='accuracy')

print("Per-fold scores:", np.round(scores, 3))
print("Mean           :", round(scores.mean(), 3))
print("Std            :", round(scores.std(), 3))
print("Spread of fold scores :", round(scores.min(), 3), "to", round(scores.max(), 3))

Per-fold scores: [0.956 0.965 0.939 0.974 0.92 ]
Mean           : 0.951
Std            : 0.019
Spread of fold scores : 0.92 to 0.974


**What just happened.** The mean over 5 folds (**0.951 ± 0.019**) is far more stable than any single holdout score, and the per-fold spread (0.920–0.974 — the same wobble we measured in Section 2) tells you how much the estimate itself can wobble. Rule of thumb: report cross-validated *mean ± spread*, never a lone number.

### The stratification trap

`KFold` does not look at the labels when it slices. That is harmless when classes are well mixed — and catastrophic when the file is *sorted by class*, which real export files very often are. Below: 500 rows where the first 50 (10%) are positives, exactly as a sorted CSV would present them.

In [4]:
from sklearn.model_selection import StratifiedKFold

y_sorted = np.zeros(500, dtype=int)
y_sorted[:50] = 1                      # 10% positives, clustered at the top
X_sorted = np.zeros((500, 1))

plain  = KFold(n_splits=5)             # no shuffle, no labels
shuffled = KFold(n_splits=5, shuffle=True, random_state=42)
strat  = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("KFold (no shuffle)   positives per fold's TEST set:",
      [int(y_sorted[te].sum()) for _, te in plain.split(X_sorted)])
print("KFold (shuffle=True) positives per fold's TEST set:",
      [int(y_sorted[te].sum()) for _, te in shuffled.split(X_sorted, y_sorted)])
print("StratifiedKFold      positives per fold's TEST set:",
      [int(y_sorted[te].sum()) for _, te in strat.split(X_sorted, y_sorted)])

KFold (no shuffle)   positives per fold's TEST set: [50, 0, 0, 0, 0]
KFold (shuffle=True) positives per fold's TEST set: [11, 15, 4, 12, 8]
StratifiedKFold      positives per fold's TEST set: [10, 10, 10, 10, 10]


**What just happened.** With plain `KFold` on the sorted file, the first fold's *test* set contains **all 50 positives** — meaning the other four folds' *training* sets contain **zero positives**, so four of your five models never saw the positive class at all. Shuffling helps but still leaves folds with 4–15 positives instead of 10. `StratifiedKFold` guarantees **exactly 10 per fold**, every time. Use it by default for classification; that is why `cross_val_score(model, X, y, cv=5)` on a classifier silently stratifies — but the moment you pass an explicit `KFold`, that protection is off.

## 4. Leave-One-Out Cross-Validation (LOOCV)

A special case of K-Fold where *k* equals the number of data points. Each iteration trains on *n-1* samples and tests on 1 sample.

In [5]:
from sklearn.model_selection import LeaveOneOut

rs = np.random.RandomState(42)
X_small = np.sort(rs.uniform(-3, 3, 40)).reshape(-1, 1)
y_small = np.sin(X_small.ravel()) + rs.normal(0, 0.2, 40)

from sklearn.linear_model import LinearRegression
lr = LinearRegression()

loo = LeaveOneOut()
loo_mse = -cross_val_score(lr, X_small, y_small, cv=loo,
                           scoring='neg_mean_squared_error').mean()

kf5 = KFold(n_splits=5, shuffle=True, random_state=42)
fold_mse = -cross_val_score(lr, X_small, y_small, cv=kf5,
                            scoring='neg_mean_squared_error')

print(f"LOOCV  : {loo.get_n_splits(X_small)} model fits | MSE = {loo_mse:.4f}")
print(f"5-fold : 5 model fits            | MSE = {fold_mse.mean():.4f}")

LOOCV  : 40 model fits | MSE = 0.2053
5-fold : 5 model fits            | MSE = 0.1983


**What just happened.** On this 40-row dataset LOOCV needed **40 fits** for a nearly identical estimate to 5-fold's **5 fits** (MSE 0.2053 vs 0.1983). On real datasets that is *n* model fits instead of *k* — harmless for linear regression, brutal for slow models. And because each fit's training set is nearly identical, the *n* test errors are highly correlated, so the estimate's precision improves far less than the fit count suggests. Use LOOCV for tiny datasets (n < ~100) where dropping 20% of rows per fold would change the model too much.

## 5. Time Series Cross-Validation

For time-dependent data, regular shuffling is not possible. Instead, the dataset is split sequentially, ensuring training always precedes testing.

Below: a random walk of 240 steps, forecast one step ahead from its last 3 values — first honestly (walk-forward), then with a shuffled `KFold`, which lets the model secretly train on the *future* of the rows it is tested on.

In [6]:
from sklearn.model_selection import TimeSeriesSplit

rs = np.random.RandomState(7)
walk = 100 + np.cumsum(rs.normal(0.05, 1.0, size=240))    # drift + noise

# features: last 3 values -> predict the next value
X_time = np.array([walk[i - 3:i] for i in range(3, len(walk))])
y_time = walk[3:]

tscv = TimeSeriesSplit(n_splits=5)
honest = cross_val_score(LinearRegression(), X_time, y_time, cv=tscv)

leaky = cross_val_score(LinearRegression(), X_time, y_time,
                        cv=KFold(n_splits=5, shuffle=True, random_state=1))

print("Honest walk-forward  R2 per fold:", np.round(honest, 3))
print("Shuffled KFold       R2 per fold:", np.round(leaky, 3))
print("Train sizes used by walk-forward:",
      [tr.shape[0] for tr, _ in tscv.split(X_time)])

Honest walk-forward  R2 per fold: [ 0.609  0.611  0.955 -0.467  0.779]
Shuffled KFold       R2 per fold: [0.935 0.903 0.926 0.932 0.936]
Train sizes used by walk-forward: [42, 81, 120, 159, 198]


**What just happened.** The shuffled version reports a rock-steady **R² ≈ 0.93** — a fantasy, because folds let the model train on points *from the future*. The honest walk-forward splits show the truth: **0.61, 0.61, 0.96, −0.47, 0.78** — the walk is mostly noise, and one fold (−0.47) was forecast *worse than predicting the mean*. That negative fold is not a bug; it is the estimate doing its job by refusing to flatter the model. Shuffling temporal data is a form of **data leakage** — covered in depth in the [Data Leakage](../16.%20Data%20Leakage/Data%20Leakage.ipynb) note.

### Summary:

| Method | Best for |
|--------|---------|
| Holdout | Large datasets, fast evaluation |
| K-Fold | Balanced performance estimation |
| LOOCV | Small datasets, high variance |
| Time Series | Temporal data |

Cross-validation ensures robust model evaluation and selection, improving generalization to new data. The three measured rules to take away: a single split can wander **±2 points** on the same model, classification folds should be **stratified**, and temporal data must be split **forward in time** — the moment you average over shuffled time, you are measuring leakage, not performance.